# FAB-Torch 2D LJ — Model Visualization

Load a trained flow from a fab-torch `run_LJ_2D` experiment and visualize it in the
same style as `boltzmann_generators_2d/Library/visual.py`.

**Layout**
1. Configuration (paths, device, sample count)
2. Imports
3. Load config + build target (`LJParticles2D`)
4. Reconstruct flow and load checkpoint
5. Generate flow samples
6. Visualization
   - 6.1 Position density scatter
   - 6.2 2-D solvent density heatmap
   - 6.3 Solute–solvent RDF
   - 6.4 Pairwise distance distributions
   - 6.5 Energy distributions
   - 6.6 Snapshot gallery (circle view)
   - 6.7 Animated trajectory *(optional)*
   - 6.8 Direct calls to visual.py *(optional)*


## 1. Configuration

In [ ]:
import os

# ── REQUIRED ────────────────────────────────────────────────────────────────
# Directory that contains `config.yaml` and `model_checkpoints/`.
# This is the Hydra output folder produced by run_LJ_2D.py.
RUN_DIR = "/path/to/your/run"   # <-- EDIT THIS

# ── OPTIONAL ────────────────────────────────────────────────────────────────
# Path to the MD reference trajectory (.h5 or .pt) for comparison plots.
# Set to None to skip MD comparison.
MD_REF_PATH = None   # e.g. "/path/to/traj_lj_particles.h5"

# Number of flow samples to generate
N_SAMPLES = 2000
BATCH_SIZE = 256     # reduce if running out of memory

# Device
DEVICE = "cpu"       # "cuda" if a GPU is available

# ── Paths to the two projects ────────────────────────────────────────────────
# visual.py utility library
VISUAL_LIB_PATH = "/Users/fleurdolmans/Documents/UvA/Master_AI/Thesis/boltzmann_generators_2d/Library"

# fab-torch root (only needed when NOT installed as a package)
FAB_TORCH_PATH = os.path.abspath(os.getcwd())


## 2. Imports

In [ ]:
import sys
import re
import warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc, animation
from matplotlib.patches import Circle
from matplotlib.colors import LogNorm

mpl.rcParams["animation.html"] = "jshtml"
rc("font", **{"family": "serif"})
rc("mathtext", **{"default": "regular"})

# Add roots to sys.path if the packages are not installed
for _p in [FAB_TORCH_PATH, VISUAL_LIB_PATH]:
    if _p and _p not in sys.path:
        sys.path.insert(0, _p)

# fab-torch
from omegaconf import OmegaConf
from fab.target_distributions.solute_in_water_LJ_2D import LJParticles2D
from experiments.make_flow import make_lj_flow

# visual.py — optional; graceful fallback if not importable
try:
    import visual as vis
    HAS_VISUAL = True
    print("visual.py loaded from", VISUAL_LIB_PATH)
except ImportError:
    HAS_VISUAL = False
    print("visual.py not found — built-in plots will be used.")

print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available())


## 3. Load config and build target

In [ ]:
# Hydra writes config.yaml directly inside the run folder;
# some versions also place it under .hydra/.
for _candidate in [
    os.path.join(RUN_DIR, "config.yaml"),
    os.path.join(RUN_DIR, ".hydra", "config.yaml"),
]:
    if os.path.exists(_candidate):
        config_path = _candidate
        break
else:
    raise FileNotFoundError(f"config.yaml not found under {RUN_DIR}")

cfg = OmegaConf.load(config_path)
print(OmegaConf.to_yaml(cfg))


In [ ]:
# Build the target.  val_samples_path is used only for comparison plots;
# if MD_REF_PATH is None the target is built without any reference data.
target = LJParticles2D(
    dim=int(cfg.target.cartesian_dim),
    n_solvent=int(cfg.target.n_solvent),
    temperature=float(cfg.target.temperature),
    energy_cut=float(cfg.target.energy_cut),
    energy_max=float(cfg.target.energy_max),
    device=DEVICE,
    val_samples_path=MD_REF_PATH,
    box_length_nm=float(cfg.target.box_length_nm),
    solvent_sigma_nm=float(cfg.target.solvent_sigma_nm),
    solvent_epsilon_kjmol=float(cfg.target.solvent_epsilon_kjmol),
    solute_sigma_nm=float(cfg.target.solute_sigma_nm),
    solute_epsilon_kjmol=float(cfg.target.solute_epsilon_kjmol),
    n_solute=int(OmegaConf.select(cfg, "target.n_solute", default=1)),
    transform_version=str(cfg.target.transform_version),
)

# Convenient global constants
L        = float(target.box_length_nm)     # box side length (nm)
L_HALF   = L / 2.0                         # half-box  ≡  l_box in visual.py
SIGMA    = float(target.solvent_sigma_nm)  # LJ particle diameter (nm)
N_SOLUTE = int(target.n_solute)
N_SOLV   = int(target.n_solvent)
N_PART   = int(target.n_particles)

print(f"System : {N_SOLUTE} solute + {N_SOLV} solvent  |  box = {L:.3f} nm")
print(f"Internal dim : {target.internal_dim}   Cartesian dim : {target.cartesian_dim}")


## 4. Reconstruct flow and load checkpoint

In [ ]:
# Mirror the dtype used during training
if OmegaConf.select(cfg, "training.use_64_bit", default=False):
    torch.set_default_dtype(torch.float64)
    target = target.double()

# Rebuild the exact same architecture via the same factory used in training
flow = make_lj_flow(cfg=cfg, target=target)
print(flow)


In [ ]:
# Locate the highest-numbered iter_* checkpoint
chkpts_dir = os.path.join(RUN_DIR, "model_checkpoints")
assert os.path.isdir(chkpts_dir), f"model_checkpoints/ not found in {RUN_DIR}"

iter_dirs = []
for _e in os.scandir(chkpts_dir):
    if _e.is_dir():
        _m = re.search(r"iter_(\d+)$", _e.name)
        if _m:
            iter_dirs.append((_e.path, int(_m.group(1))))

assert iter_dirs, "No iter_* checkpoint folders found!"
chkpt_dir, chkpt_iter = max(iter_dirs, key=lambda x: x[1])
model_path = os.path.join(chkpt_dir, "model.pt")
print(f"Loading iteration {chkpt_iter} checkpoint: {model_path}")

checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
flow.load_state_dict(checkpoint["flow"])
flow = flow.to(DEVICE)
flow.eval()
print("Weights loaded.")


## 5. Generate flow samples

In [ ]:
# Sample in internal space, evaluate energy, transform to Cartesian
_x_cart_list, _lp_list = [], []

with torch.no_grad():
    n_done = 0
    while n_done < N_SAMPLES:
        n_now = min(BATCH_SIZE, N_SAMPLES - n_done)
        z, _  = flow.sample_and_log_prob((n_now,))
        x_c, _= target.coordinate_transform.forward(z)
        lp    = target.log_prob(z)
        _x_cart_list.append(x_c.detach().cpu())
        _lp_list.append(lp.detach().cpu())
        n_done += n_now
        if n_done % 500 == 0 or n_done == N_SAMPLES:
            print(f"  {n_done}/{N_SAMPLES}")

# x_flow_flat : (N, 2*n_part)  flat Cartesian in nm
# x_flow      : (N, n_part, 2)
x_flow_flat = torch.cat(_x_cart_list).float()                   # (N, 2*n_part)
x_flow      = x_flow_flat.view(N_SAMPLES, N_PART, 2).numpy()    # (N, n_part, 2)
u_flow      = (-torch.cat(_lp_list).float()).numpy()             # reduced energy

print(f"\nFlow samples : {x_flow.shape}")
print(f"Energy  mean={u_flow.mean():.2f}  std={u_flow.std():.2f}  "
      f"finite={np.isfinite(u_flow).mean()*100:.1f}%")


In [ ]:
# Load MD reference data (only if MD_REF_PATH was set)
if target.val_data_x is not None:
    x_md_flat = target.val_data_x.float().cpu()           # (M, 2*n_part)
    x_md      = x_md_flat.view(-1, N_PART, 2).numpy()     # (M, n_part, 2)
    with torch.no_grad():
        lp_md = target.log_prob(
            target.val_data_i.float().to(DEVICE)
        ).cpu()
    u_md  = (-lp_md.float()).numpy()
    print(f"MD reference : {x_md.shape}")
    print(f"Energy  mean={u_md.mean():.2f}  std={u_md.std():.2f}")
else:
    x_md, u_md = None, None
    print("No MD reference data loaded.")


## 6. Visualization

Colour convention throughout (matching visual.py):
- **orangered** – solute (particle 0)
- **steelblue** – solvent (particles 1 …)
- dashed grey rectangle – periodic box boundary

### 6.1  Position density scatter

In [ ]:
def _scatter_density(ax, traj, subtitle, l_box,
                     solute_color="orangered", solvent_color="steelblue"):
    """Scatter all frames — mirrors visual.py position_density."""
    ax.scatter(
        traj[:, N_SOLUTE:, 0].ravel(), traj[:, N_SOLUTE:, 1].ravel(),
        s=1, c=solvent_color, alpha=0.04, label="Solvent",
    )
    ax.scatter(
        traj[:, 0, 0], traj[:, 0, 1],
        s=2, c=solute_color, alpha=0.30, label="Solute",
    )
    ax.add_patch(plt.Rectangle(
        (-l_box, -l_box), 2 * l_box, 2 * l_box,
        fill=False, ec="gray", ls="--",
    ))
    ax.set_xlim(-l_box - 0.3, l_box + 0.3)
    ax.set_ylim(-l_box - 0.3, l_box + 0.3)
    ax.set_aspect("equal")
    ax.set_title(subtitle, fontsize=16)
    ax.xaxis.set_visible(False)
    ax.yaxis.set_visible(False)


_datasets = [(x_flow, "Flow samples")]
if x_md is not None:
    _datasets.insert(0, (x_md, "MD reference"))

fig, axes = plt.subplots(1, len(_datasets), figsize=(5 * len(_datasets), 5))
if len(_datasets) == 1:
    axes = [axes]

for _ax, (_traj, _sub) in zip(axes, _datasets):
    _scatter_density(_ax, _traj, _sub, l_box=L_HALF)

axes[-1].legend(markerscale=6, loc="upper right", fontsize=12)
fig.suptitle("Position density  (red = solute, blue = solvent)", fontsize=18)
plt.tight_layout()
plt.show()


### 6.2  2D solvent density heatmap

In [ ]:
def _density_hist2d(traj, n_bins=80, l_box=None):
    """2D histogram of solvent positions + spatial entropy."""
    solv = traj[:, N_SOLUTE:, :].reshape(-1, 2)
    H, _, _ = np.histogram2d(
        solv[:, 0], solv[:, 1],
        bins=n_bins,
        range=[[-l_box, l_box], [-l_box, l_box]],
    )
    p = H / H.sum()
    entropy = float(-np.sum(p[p > 0] * np.log(p[p > 0])))
    return H, entropy


H_flow, S_flow = _density_hist2d(x_flow, l_box=L_HALF)

_panels = [(H_flow, f"Flow  (H = {S_flow:.4f})")]
if x_md is not None:
    H_md, S_md = _density_hist2d(x_md, l_box=L_HALF)
    _panels.insert(0, (H_md, f"MD  (H = {S_md:.4f})"))

# Shared log-scale colour limits
_pos = np.concatenate([h[h > 0].ravel() for h, _ in _panels])
_vmin = max(1, np.percentile(_pos, 1))
_vmax = np.percentile(_pos, 99.5)
if _vmax <= _vmin:
    _vmax = _pos.max()
_norm = LogNorm(vmin=_vmin, vmax=_vmax)

fig, axes = plt.subplots(
    1, len(_panels), figsize=(5.5 * len(_panels), 4.8),
    constrained_layout=True,
)
if len(_panels) == 1:
    axes = [axes]

_last_im = None
for _ax, (H, _title) in zip(axes, _panels):
    _last_im = _ax.imshow(
        H.T, origin="lower",
        extent=(-L_HALF, L_HALF, -L_HALF, L_HALF),
        aspect="equal", cmap="magma", norm=_norm,
    )
    _ax.scatter([0], [0], s=35, c="cyan", edgecolors="black",
                linewidths=0.5, label="solute", zorder=3)
    _ax.add_patch(plt.Rectangle(
        (-L_HALF, -L_HALF), 2 * L_HALF, 2 * L_HALF,
        fill=False, ec="white", ls="--", lw=1.0, alpha=0.7,
    ))
    _ax.set_title(_title, fontsize=16)
    _ax.set_xlabel("x (nm)", fontsize=12)
    _ax.set_ylabel("y (nm)", fontsize=12)
    _ax.legend(loc="upper right", frameon=True, fontsize=12)

fig.colorbar(_last_im, ax=axes, shrink=0.9).set_label(
    "solvent count per bin (log scale)", fontsize=12
)
plt.suptitle("2D solvent density", fontsize=18)
plt.show()


### 6.3  Solute–solvent RDF

In [ ]:
def compute_rdf_2d(traj_flat, n_solute, n_solvent, L, dr=0.005):
    """
    2-D radial distribution function g(r) between solute and solvent
    with minimum-image convention.

    Parameters
    ----------
    traj_flat : ndarray (N, 2*n_particles)  Cartesian positions in nm
    """
    N  = traj_flat.shape[0]
    X  = traj_flat.reshape(N, -1, 2)
    sol  = X[:, :n_solute, :]                        # (N, n_s, 2)
    solv = X[:, n_solute:n_solute + n_solvent, :]    # (N, n_v, 2)

    diff = solv[:, None, :, :] - sol[:, :, None, :]  # (N, n_s, n_v, 2)
    diff -= L * np.round(diff / L)
    r = np.linalg.norm(diff, axis=-1).ravel()        # (N * n_s * n_v,)

    r_max  = 0.5 * L
    n_bins = int(np.floor(r_max / dr))
    edges  = np.linspace(0.0, n_bins * dr, n_bins + 1)
    counts, _ = np.histogram(r, bins=edges)

    r_c      = 0.5 * (edges[:-1] + edges[1:])
    shell    = 2.0 * np.pi * r_c * dr       # 2-D annulus area
    rho      = n_solvent / (L ** 2)
    expected = N * n_solute * rho * shell

    g_r = counts / np.maximum(expected, 1e-12)
    g_r[0] = 0.0
    return r_c, g_r


r_flow, g_flow = compute_rdf_2d(x_flow_flat.numpy(), N_SOLUTE, N_SOLV, L)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r_flow, g_flow, lw=2, color="steelblue", label="Flow")

if x_md is not None:
    r_md, g_md = compute_rdf_2d(
        x_md.reshape(len(x_md), -1), N_SOLUTE, N_SOLV, L
    )
    ax.plot(r_md, g_md, lw=2, color="black", label="MD reference")
else:
    r_md = g_md = None

ax.axhline(1.0, color="k", ls="--", lw=0.8, label="Ideal gas")
ax.axvline(SIGMA, color="gray", ls=":", lw=0.8, label=f"σ = {SIGMA:.2f} nm")
ax.set_xlabel("r  (nm)", fontsize=12)
ax.set_ylabel("g(r)", fontsize=12)
ax.set_title("Solute–solvent RDF", fontsize=18)
ax.set_xlim(0, 0.5 * L)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


### 6.4  Pairwise distance distributions

In [ ]:
def min_pairwise_dist(X, L, mode="ss", n_solute=1):
    """
    Per-sample minimum pairwise distance (PBC-corrected).

    Parameters
    ----------
    X    : ndarray (N, n_particles, 2)
    mode : 'ss'  solvent–solvent  |  'su'  solvent–solute

    Returns ndarray (N,)
    """
    solute  = X[:, :n_solute, :]    # (N, n_sol, 2)
    solvent = X[:, n_solute:, :]    # (N, n_solv, 2)

    if mode == "ss":
        diff = solvent[:, :, None, :] - solvent[:, None, :, :]  # (N, n_v, n_v, 2)
        diff -= L * np.round(diff / L)
        d    = np.linalg.norm(diff, axis=-1)                    # (N, n_v, n_v)
        # mask self-pairs
        eye = np.eye(d.shape[1], dtype=bool)[None, :, :]
        d   = np.where(eye, np.inf, d)
        return d.min(axis=(1, 2))

    elif mode == "su":
        diff = solvent[:, :, None, :] - solute[:, None, :, :]   # (N, n_v, n_sol, 2)
        diff -= L * np.round(diff / L)
        d    = np.linalg.norm(diff, axis=-1)                    # (N, n_v, n_sol)
        return d.min(axis=(1, 2))

    else:
        raise ValueError(f"Unknown mode '{mode}'")


d_ss_flow = min_pairwise_dist(x_flow, L, mode="ss", n_solute=N_SOLUTE)
d_su_flow = min_pairwise_dist(x_flow, L, mode="su", n_solute=N_SOLUTE)

_dist_pairs = [("Flow", d_ss_flow, d_su_flow)]
if x_md is not None:
    d_ss_md = min_pairwise_dist(x_md, L, mode="ss", n_solute=N_SOLUTE)
    d_su_md = min_pairwise_dist(x_md, L, mode="su", n_solute=N_SOLUTE)
    _dist_pairs.insert(0, ("MD reference", d_ss_md, d_su_md))

_colors = ["black", "steelblue", "orangered", "purple"]
_all    = np.concatenate([d for _, d_ss, d_su in _dist_pairs for d in [d_ss, d_su]])
_bins   = np.linspace(0, np.nanmax(_all[np.isfinite(_all)]), 80)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for _ax, (_col_idx, _dist_key, _stitle) in zip(
    axes,
    [
        (1, "ss", "Solvent–solvent  (min pairwise distance)"),
        (2, "su", "Solvent–solute   (min pairwise distance)"),
    ],
):
    for _i, (_lbl, _d_ss, _d_su) in enumerate(_dist_pairs):
        _d = _d_ss if _dist_key == "ss" else _d_su
        _ax.hist(
            _d, bins=_bins, density=True, alpha=0.30,
            color=_colors[_i % len(_colors)], label=_lbl,
            histtype="stepfilled",
        )
    _ax.axvline(SIGMA, color="k", ls="--", lw=1.2, label=f"σ = {SIGMA:.2f} nm")
    _ax.set_xlabel("Min distance  (nm)", fontsize=16)
    _ax.set_ylabel("Density", fontsize=16)
    _ax.set_title(_stitle, fontsize=16)
    _ax.legend(fontsize=12, loc="upper right")

plt.tight_layout()
plt.show()


### 6.5  Energy distributions

In [ ]:
_finite_flow = u_flow[np.isfinite(u_flow)]
x_max_e = float(np.percentile(_finite_flow, 99))
if u_md is not None:
    x_max_e = max(x_max_e, float(np.percentile(u_md[np.isfinite(u_md)], 99)))

_energy_sets = [(u_flow, "Flow", "steelblue")]
if u_md is not None:
    _energy_sets.insert(0, (u_md, "MD reference", "black"))

fig, ax = plt.subplots(figsize=(7, 4))
for _e, _lbl, _c in _energy_sets:
    _ep   = _e[_e < x_max_e]
    _frac = len(_ep) / len(_e) * 100
    ax.hist(_ep, bins=80, density=True, alpha=0.35, color=_c,
            label=f"{_lbl} ({_frac:.0f}% shown)")

ax.set_xlabel("Reduced energy  (−log p)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Energy distribution", fontsize=18)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


### 6.6  Snapshot gallery  (circle view)

In [ ]:
# Renders individual configurations with Circle patches,
# matching the style of visual.py make_2D_traj_circles.

N_SNAP = 4   # how many snapshots to show per source

def draw_snapshot(ax, pos, title=None):
    radius = SIGMA / 2.0
    ax.set_xlim(-L_HALF - 0.5, L_HALF + 0.5)
    ax.set_ylim(-L_HALF - 0.5, L_HALF + 0.5)
    ax.set_aspect("equal")
    ax.add_patch(plt.Rectangle(
        (-L_HALF, -L_HALF), 2 * L_HALF, 2 * L_HALF,
        fill=False, edgecolor="gray", linestyle="--", linewidth=1.2,
    ))
    for p, xy in enumerate(pos):
        color = "orangered" if p < N_SOLUTE else "steelblue"
        alpha = 1.0          if p < N_SOLUTE else 0.5
        ax.add_patch(
            Circle(xy, radius=radius, facecolor=color,
                   edgecolor="black", alpha=alpha)
        )
    if title:
        ax.set_title(title, fontsize=12)
    ax.set_xlabel("x (nm)", fontsize=9)
    ax.set_ylabel("y (nm)", fontsize=9)


_sources = [(x_flow, "Flow")]
if x_md is not None:
    _sources.insert(0, (x_md, "MD"))

_ncols = N_SNAP * len(_sources)
fig, axes = plt.subplots(1, _ncols, figsize=(3.5 * _ncols, 3.5))
if _ncols == 1:
    axes = [axes]

_k = 0
for _src, _src_lbl in _sources:
    for _n in range(N_SNAP):
        draw_snapshot(axes[_k], _src[_n], title=f"{_src_lbl} #{_n + 1}")
        _k += 1

plt.suptitle("Configuration snapshots", fontsize=18)
plt.tight_layout()
plt.show()


### 6.7  Animated trajectory  *(optional)*

In [ ]:
# Animate a sequence of flow samples as if they were MD frames.
# Delegates to visual.py when available, otherwise uses FuncAnimation directly.

N_ANIM = min(200, N_SAMPLES)
x_anim = x_flow[:N_ANIM]     # (N_ANIM, n_part, 2)

if HAS_VISUAL:
    fig, ani = vis.make_2D_traj_circles(
        x_anim,
        box=(L, L),
        sigma=SIGMA,
        fps=10,
        animate=True,
        title="Flow samples — 2D LJ",
    )
else:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(-L_HALF - 0.5, L_HALF + 0.5)
    ax.set_ylim(-L_HALF - 0.5, L_HALF + 0.5)
    ax.set_aspect("equal")
    ax.add_patch(plt.Rectangle(
        (-L_HALF, -L_HALF), 2 * L_HALF, 2 * L_HALF,
        fill=False, ec="gray", ls="--",
    ))
    _circles = []
    for _p in range(N_PART):
        _c = Circle(
            x_anim[0, _p], radius=SIGMA / 2,
            facecolor="orangered" if _p < N_SOLUTE else "steelblue",
            edgecolor="black",
            alpha=1.0 if _p < N_SOLUTE else 0.5,
        )
        ax.add_patch(_c)
        _circles.append(_c)

    def _upd(frame):
        for _p, _c in enumerate(_circles):
            _c.center = tuple(x_anim[frame, _p])
        return _circles

    ani = animation.FuncAnimation(
        fig, _upd, frames=N_ANIM, interval=100, blit=True
    )

from IPython.display import display
display(ani)


### 6.8  Direct calls to visual.py  *(optional)*

Re-uses the arrays computed above and passes them straight to the visual.py functions.

> **Note:** `visualize_rdf` in visual.py references an undefined `zoom` variable — this cell
> catches that error and falls back to the built-in RDF plot.

In [ ]:
if not HAS_VISUAL:
    print("visual.py not available — skipping this cell.")
else:
    # ── RDF ──────────────────────────────────────────────────────────────
    _r_list  = [r_flow] + ([r_md]  if r_md  is not None else [])
    _g_list  = [g_flow] + ([g_md]  if g_md  is not None else [])
    _lbl_rdf = ["Flow"]  + (["MD reference"] if r_md is not None else [])

    try:
        vis.visualize_rdf(
            r=_r_list, gr=_g_list, label=_lbl_rdf,
            sigma=SIGMA, title="Solute–solvent RDF",
        )
    except NameError as _e:
        # visual.py has a known bug: 'zoom' is referenced but never defined.
        warnings.warn(f"visualize_rdf raised NameError ({_e}); using built-in plot.")
        fig, ax = plt.subplots(figsize=(8, 5))
        for _r, _g, _lbl, _col in zip(_r_list, _g_list, _lbl_rdf, ["steelblue", "black"]):
            ax.plot(_r, _g, lw=2, color=_col, label=_lbl)
        ax.axhline(1, color="k", ls="--", lw=0.8, label="Ideal gas")
        ax.axvline(SIGMA, color="gray", ls=":", lw=0.8, label=f"σ = {SIGMA:.2f} nm")
        ax.set_xlabel("r (nm)"); ax.set_ylabel("g(r)")
        ax.set_title("Solute–solvent RDF", fontsize=18)
        ax.legend(); plt.tight_layout(); plt.show()

    # ── Pairwise distances ────────────────────────────────────────────────
    _ss_in = [d_ss_flow] + ([d_ss_md] if x_md is not None else [])
    _su_in = [d_su_flow] + ([d_su_md] if x_md is not None else [])
    _lbls  = ["Flow"]    + (["MD reference"] if x_md is not None else [])

    vis.visualize_distances(
        ss=_ss_in, vv=_su_in, label=_lbls,
        sigma=SIGMA, measure="min",
        sub_titles=[
            "Solvent–solute  (particle 0 vs rest)",
            "Solvent–solvent (all solvent pairs)",
        ],
    )

    # ── Energies ──────────────────────────────────────────────────────────
    _e_in  = ([u_md]           if u_md  is not None else []) + [u_flow]
    _e_lbl = (["MD reference"] if u_md  is not None else []) + ["Flow"]
    vis.plot_energies(_e_in, label=_e_lbl, x_max=float(x_max_e))

    # ── Position density ─────────────────────────────────────────────────
    _pd_sets = ([(x_md, "MD reference")] if x_md is not None else []) + \
               [(x_flow, "Flow samples")]
    vis.position_density(datasets=_pd_sets, l_box=L_HALF)
